#ED Phase-2 — Trainer (from scratch) + Isotonic Calibration\n\nPurpose: Train a small tabular model on your attached CSVs (train_DE_full.csv, val_DE_full.csv, test_DE_full.csv), calibrate probabilities with isotonic regression, enforce a recall ≥ 0.85 floor on validation for the classification threshold, evaluate on test, and save a scorer bundle for your OPS notebook.\n\nOutputs\n- ed_phase2_model.joblib — pipeline + isotonic calibrator + threshold + metadata\n- test_predictions.csv — per-row calibrated probabilities on the test set\n- Metrics printed to stdout\n\n> This notebook is self-contained and does not train inside the OPS UI notebook.

In [1]:

# --- Bootstrap: environment + paths + logger ---
import os, json, datetime as _dt
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else ("/content" if os.path.exists("/content") else "/mnt/data")
_p = lambda *p: os.path.join(BASE, *p)

CONFIG = {
    "TRAIN_PATH": _p("train_DE_full.csv"),
    "VAL_PATH":   _p("val_DE_full.csv"),
    "TEST_PATH":  _p("test_DE_full.csv"),
    "MODEL_OUT":  _p("ed_phase2_model.joblib"),
    "PRED_OUT":   _p("test_predictions.csv"),
    "EVENT_LOG_PATH": _p("event_log.jsonl"),
}
os.makedirs(BASE, exist_ok=True)
def _append_event(ev: dict):
    ev = {"ts": _dt.datetime.utcnow().isoformat()+"Z", **(ev or {})}
    with open(CONFIG["EVENT_LOG_PATH"], "a", encoding="utf-8") as f:
        f.write(json.dumps(ev, ensure_ascii=False) + "\n")
print("BASE =", BASE)
print("Outputs →", CONFIG["MODEL_OUT"], "and", CONFIG["PRED_OUT"])

# --- Lock dataset paths (runs before any resolver) ---
import os
DATASET = "testtrainval"  # your attached dataset slug

CONFIG["TRAIN_PATH"] = f"/kaggle/input/gen4-synthetic-data/gen4_de_full_units.csv"
CONFIG["VAL_PATH"]   = f"/kaggle/input/{DATASET}/val_DE_full.csv"
CONFIG["TEST_PATH"]  = f"/kaggle/input/{DATASET}/test_DE_full.csv"

# optional: tell the resolver to skip
CONFIG["SKIP_AUTODISCOVERY"] = True

# assert early, fail with a clear message if not attached
for k in ("TRAIN_PATH","VAL_PATH","TEST_PATH"):
    assert os.path.exists(CONFIG[k]), f"Attach dataset '{DATASET}' with file {os.path.basename(CONFIG[k])}"
print("Paths locked:", CONFIG["TRAIN_PATH"], CONFIG["VAL_PATH"], CONFIG["TEST_PATH"])


BASE = /kaggle/working
Outputs → /kaggle/working/ed_phase2_model.joblib and /kaggle/working/test_predictions.csv
Paths locked: /kaggle/input/gen4-synthetic-data/gen4_de_full_units.csv /kaggle/input/testtrainval/val_DE_full.csv /kaggle/input/testtrainval/test_DE_full.csv


In [2]:

# Add input path for datasets:
INPUT_BASE = "/kaggle/input"
def _input_path(dataset_name, filename):
    return os.path.join(INPUT_BASE, dataset_name, filename)

# Check available datasets:
if os.path.exists(INPUT_BASE):
    print("Available datasets:", os.listdir(INPUT_BASE))

Available datasets: ['gen4-synthetic-data', 'coding-maps', 'testtrainval']


In [3]:
# --- Force dataset paths (since they're attached as "testtrainval") ---
CONFIG["TRAIN_PATH"] = "/kaggle/input/gen4-synthetic-data/gen4_de_full_units.csv"
CONFIG["VAL_PATH"]   = "/kaggle/input/testtrainval/val_DE_full.csv"
CONFIG["TEST_PATH"]  = "/kaggle/input/testtrainval/test_DE_full.csv"

# sanity
import os
for k in ["TRAIN_PATH","VAL_PATH","TEST_PATH"]:
    p = CONFIG[k]
    print(k, "→", p, "| exists:", os.path.exists(p))


TRAIN_PATH → /kaggle/input/gen4-synthetic-data/gen4_de_full_units.csv | exists: True
VAL_PATH → /kaggle/input/testtrainval/val_DE_full.csv | exists: True
TEST_PATH → /kaggle/input/testtrainval/test_DE_full.csv | exists: True


In [4]:

# --- Load data & split into X, y (auto-detect label) ---
import pandas as pd, numpy as np

def _pick_label(df: pd.DataFrame):
    # Prioritized common names
    for cand in ["label","y","target","TARGET","outcome"]:
        if cand in df.columns:
            return cand
    # Fallback: last column
    return df.columns[-1]

def _split_xy(df: pd.DataFrame, label_name: str):
    y = df[label_name]
    X = df.drop(columns=[label_name])
    # Coerce y to binary ints if possible
    if y.dtype.kind in "biu":
        y_bin = (y > 0).astype(int).values
    else:
        yn = y.astype(str).str.lower().map({"1":1,"true":1,"t":1,"yes":1,"y":1,"positive":1,"pos":1,
                                            "0":0,"false":0,"f":0,"no":0,"n":0,"negative":0,"neg":0})
        if yn.isna().any():
            # fallback: compare to mode
            mode = y.mode().iloc[0]
            yn = (y == mode).astype(int)
        y_bin = yn.values.astype(int)
    return X, y_bin

import pandas as pd
df_tr = pd.read_csv(CONFIG["TRAIN_PATH"])
df_va = pd.read_csv(CONFIG["VAL_PATH"])
df_te = pd.read_csv(CONFIG["TEST_PATH"])

LABEL = _pick_label(df_tr)
print("Detected label column:", LABEL)

X_tr, y_tr = _split_xy(df_tr, LABEL)
X_va, y_va = _split_xy(df_va, LABEL if LABEL in df_va.columns else _pick_label(df_va))
X_te, y_te = _split_xy(df_te, LABEL if LABEL in df_te.columns else _pick_label(df_te))

# Keep only numeric columns for the quick trainer
num_cols = X_tr.select_dtypes(include=["number"]).columns.tolist()
X_tr, X_va, X_te = X_tr[num_cols], X_va[num_cols], X_te[num_cols]

print(f"Train: X={X_tr.shape}, positives={y_tr.sum()} ({y_tr.mean():.3f})")
print(f"Valid: X={X_va.shape}, positives={y_va.sum()} ({y_va.mean():.3f})")
print(f"Test : X={X_te.shape}, positives={y_te.sum()} ({y_te.mean():.3f})")


Detected label column: Gate_Pos
Train: X=(9755, 16), positives=919 (0.094)
Valid: X=(5760, 16), positives=713 (0.124)
Test : X=(5760, 16), positives=823 (0.143)


In [5]:

# --- Build & fit a small model (choose 'mlp' or 'logreg') ---
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

MODEL_KIND = "mlp"  # 'mlp' or 'logreg'

if MODEL_KIND == "mlp":
    est = MLPClassifier(hidden_layer_sizes=(32,16), activation="relu", solver="adam",
                        alpha=1e-4, learning_rate_init=1e-3, max_iter=200, random_state=42)
else:
    est = LogisticRegression(max_iter=200, class_weight="balanced", n_jobs=None, random_state=42)

pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler(with_mean=True, with_std=True)),
    ("est", est),
])

pipe.fit(X_tr, y_tr)
print("Fitted base model:", type(est).__name__)


Fitted base model: MLPClassifier


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [6]:

# --- Isotonic calibration (fit on validation) + pick threshold s.t. recall >= 0.85 ---
import numpy as np
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score, brier_score_loss, classification_report

p_va_raw = pipe.predict_proba(X_va)[:,1]

# Fit isotonic calibrator on validation raw scores
cal = IsotonicRegression(out_of_bounds="clip")
cal.fit(p_va_raw, y_va)

def calibrate(p):
    return cal.transform(np.asarray(p))

p_va_cal = calibrate(p_va_raw)

# Choose threshold: maximum precision subject to recall >= 0.85 (if unattainable, pick smallest threshold meeting recall)
prec, rec, thr = precision_recall_curve(y_va, p_va_cal)
thr_full = np.r_[0.0, thr]  # align sizes with prec/rec
mask = rec >= 0.85
if mask.any():
    idx = np.argmax(prec[mask])  # precision-maximizing index among recall≥0.85
    thr_star = float(thr_full[mask][idx])
else:
    thr_star = float(thr_full[np.argmax(rec)])
print(f"Selected threshold (recall≥0.85 floor): {thr_star:.4f}")

# Quick validation metrics
auc_val = roc_auc_score(y_va, p_va_cal)
ap_val  = average_precision_score(y_va, p_va_cal)
brier_val = brier_score_loss(y_va, p_va_cal)
yhat_va = (p_va_cal >= thr_star).astype(int)
print("Validation AUC:", round(auc_val,4), "AP:", round(ap_val,4), "Brier:", round(brier_val,4))
print("Validation report:\n", classification_report(y_va, yhat_va, digits=3))


Selected threshold (recall≥0.85 floor): 0.0174
Validation AUC: 0.5383 AP: 0.1419 Brier: 0.1073
Validation report:
               precision    recall  f1-score   support

           0      0.000     0.000     0.000      5047
           1      0.124     1.000     0.220       713

    accuracy                          0.124      5760
   macro avg      0.062     0.500     0.110      5760
weighted avg      0.015     0.124     0.027      5760



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [7]:

# --- Evaluate on test ---
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, classification_report, confusion_matrix

p_te_raw = pipe.predict_proba(X_te)[:,1]
p_te_cal = calibrate(p_te_raw)

auc_te = roc_auc_score(y_te, p_te_cal)
ap_te  = average_precision_score(y_te, p_te_cal)
brier_te = brier_score_loss(y_te, p_te_cal)
yhat_te = (p_te_cal >= thr_star).astype(int)

tn, fp, fn, tp = confusion_matrix(y_te, yhat_te).ravel()
rec_te = tp/(tp+fn) if (tp+fn)>0 else 0.0
prec_te = tp/(tp+fp) if (tp+fp)>0 else 0.0

print(f"Test  AUC: {auc_te:.4f}  AP: {ap_te:.4f}  Brier: {brier_te:.4f}")
print("Test  report:\n", classification_report(y_te, yhat_te, digits=3))
print(f"Test  confusion: TN={tn} FP={fp} FN={fn} TP={tp}  (recall={rec_te:.3f}, precision={prec_te:.3f})")


Test  AUC: 0.5095  AP: 0.1763  Brier: 0.1225
Test  report:
               precision    recall  f1-score   support

           0      0.000     0.000     0.000      4937
           1      0.143     1.000     0.250       823

    accuracy                          0.143      5760
   macro avg      0.071     0.500     0.125      5760
weighted avg      0.020     0.143     0.036      5760

Test  confusion: TN=0 FP=4937 FN=0 TP=823  (recall=1.000, precision=0.143)


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


In [8]:

# --- Save artifacts ---
from joblib import dump
import pandas as pd, json

bundle = {
    "pipeline": pipe,
    "calibrator": cal,
    "threshold": float(thr_star),
    "features": list(X_tr.columns),
    "label": str(LABEL),
    "model_kind": "MLP" if hasattr(pipe.named_steps["est"], "hidden_layer_sizes") else "LogReg",
    "created_utc": _dt.datetime.utcnow().isoformat()+"Z",
}

dump(bundle, CONFIG["MODEL_OUT"])
print("Saved:", CONFIG["MODEL_OUT"])

# Write test predictions
pd.DataFrame({"prob_cal": p_te_cal, "y_true": y_te}).to_csv(CONFIG["PRED_OUT"], index=False)
print("Saved:", CONFIG["PRED_OUT"])

_append_event({"type":"trainer_complete",
               "model_out": CONFIG["MODEL_OUT"],
               "pred_out": CONFIG["PRED_OUT"],
               "threshold": float(thr_star)})
print("Logged to:", CONFIG["EVENT_LOG_PATH"])


Saved: /kaggle/working/ed_phase2_model.joblib
Saved: /kaggle/working/test_predictions.csv
Logged to: /kaggle/working/event_log.jsonl


In [ ]:

# --- How to use in OPS notebook ---
from joblib import load
import pandas as pd

def load_scorer(bundle_path=CONFIG["MODEL_OUT"]):
    b = load(bundle_path)
    pipe = b["pipeline"]
    cal  = b["calibrator"]
    thr  = b["threshold"]
    feats= b["features"]
    def score_proba(df: pd.DataFrame):
        X = df[feats]
        p_raw = pipe.predict_proba(X)[:,1]
        import numpy as np
        return cal.transform(np.asarray(p_raw))
    return score_proba, thr, feats, b

score_proba, thr, feats, meta = load_scorer()
print("Loaded scorer. Threshold =", thr, "Features:", len(feats))


In [10]:
# Paths (adjust only if your files live elsewhere)
SYN_PATH  = "/kaggle/input/gen4-synthetic-data/gen4_de_full_units.csv"  
VAL_PATH  = "/kaggle/input/testtrainval/val_DE_full.csv"
TEST_PATH = "/kaggle/input/testtrainval/test_DE_full.csv"
CODING_MAPS_PATH = "/kaggle/input/coding-maps/coding_maps_json.json"  
OUT_DIR  = "/kaggle/working/ed_synth100_artifacts"

import os, json, pandas as pd, numpy as np, textwrap, time, sys, pathlib
from pathlib import Path

# Load
df_train = pd.read_csv(SYN_PATH)
df_val   = pd.read_csv(VAL_PATH)
df_test  = pd.read_csv(TEST_PATH)
coding_maps = json.loads(Path(CODING_MAPS_PATH).read_text())

# Schema expected (16 features + target)
FEATURES = ["Tag","t_min","Triage","Leitsymptom","HF","MAP","ICU_Kap","Kap_veraltet","t_norm",
            "hat_Labor","Labor_ausstehend","hat_Roentgen","Roentgen_ausstehend","hat_CT","CT_ausstehend",
            "naechste_Aktion"]
TARGET = "Gate_Pos"

# Invariant checks (evidence-producing)
for df_name, df in [("train", df_train), ("val", df_val), ("test", df_test)]:
    missing = sorted([c for c in FEATURES+[TARGET] if c not in df.columns])
    assert not missing, f"{df_name} missing columns: {missing}"

assert set(df_train.columns) >= set(FEATURES+[TARGET]), "train schema mismatch"
assert set(df_val.columns)   >= set(FEATURES+[TARGET]),   "val schema mismatch"
assert set(df_test.columns)  >= set(FEATURES+[TARGET]),   "test schema mismatch"

print("OK: schemas validated.")
print("Train size:", df_train.shape, "Val size:", df_val.shape, "Test size:", df_test.shape)
print("Train prevalence Gate_Pos:", df_train[TARGET].mean())
print("Val prevalence Gate_Pos:",   df_val[TARGET].mean())
print("Test prevalence Gate_Pos:",  df_test[TARGET].mean())
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)


OK: schemas validated.
Train size: (9755, 18) Val size: (5760, 18) Test size: (5760, 18)
Train prevalence Gate_Pos: 0.09420809841107125
Val prevalence Gate_Pos: 0.12378472222222223
Test prevalence Gate_Pos: 0.14288194444444444


In [11]:
import joblib, time
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

X_tr, y_tr = df_train[FEATURES].values, df_train[TARGET].values.astype(int)

# Architecture mirrors a small 2-layer ReLU MLP; 100 epochs (scikit-learn: max_iter=100).
mlp = MLPClassifier(
    hidden_layer_sizes=(32,16),  # keep slim to avoid slowdowns; still nonlinear
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=100,                # <-- 100 epochs
    random_state=42,
    early_stopping=False,
    warm_start=False,
    verbose=False
)

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("mlp", mlp)
])

t0 = time.time()
pipe.fit(X_tr, y_tr)
print(f"Fit done in {time.time()-t0:.2f}s")
# Smoke: confirm loss curve exists (sklearn MLP stores it)
assert hasattr(pipe.named_steps["mlp"], "loss_curve_") and len(pipe.named_steps["mlp"].loss_curve_) >= 1
print("OK: 100-epoch training complete (loss_curve_ present).")

joblib.dump(pipe, f"{OUT_DIR}/ed_phase2_model_synth100.joblib")
print("Saved model:", f"{OUT_DIR}/ed_phase2_model_synth100.joblib")


Fit done in 3.69s
OK: 100-epoch training complete (loss_curve_ present).
Saved model: /kaggle/working/ed_synth100_artifacts/ed_phase2_model_synth100.joblib


/usr/local/lib/python3.11/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:686: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


In [12]:
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score, f1_score, confusion_matrix

X_val, y_val = df_val[FEATURES].values,  df_val[TARGET].values.astype(int)
X_te,  y_te  = df_test[FEATURES].values, df_test[TARGET].values.astype(int)

# 1) Calibrate probabilities (isotonic) on real validation
p_val_raw = pipe.predict_proba(X_val)[:,1]
p_te_raw  = pipe.predict_proba(X_te)[:,1]

iso = IsotonicRegression(y_min=0, y_max=1, out_of_bounds="clip")
iso.fit(p_val_raw, y_val)

# Smoke: isotonic is monotone and bounded
p_chk = iso.transform(np.array([0.0, 0.25, 0.5, 0.75, 1.0]))
assert np.all(np.diff(p_chk) >= -1e-8)
assert (p_chk.min() >= -1e-8) and (p_chk.max() <= 1+1e-8)
print("OK: isotonic fitted & monotone.")

p_val = iso.transform(p_val_raw)
p_te  = iso.transform(p_te_raw)

# 2) Threshold search to achieve recall ≥ 0.85 on validation
thr_candidates = np.unique(np.round(p_val, 6))
thr_candidates.sort()
best_thr = None
for t in thr_candidates:
    if recall_score(y_val, (p_val >= t).astype(int), zero_division=0) >= 0.85:
        best_thr = float(t); break
if best_thr is None:
    # fallback: pick smallest threshold that gives max recall
    recalls = [(t, recall_score(y_val, (p_val >= t).astype(int), zero_division=0)) for t in thr_candidates]
    t_max = max(recalls, key=lambda x: x[1])[0]
    best_thr = float(t_max)

# Smoke assert: actually meets the floor on validation
val_recall = recall_score(y_val, (p_val >= best_thr).astype(int), zero_division=0)
assert val_recall >= 0.85 - 1e-9, f"Recall floor not met at selected threshold: {val_recall:.3f}"
print(f"OK: recall floor met on validation @thr={best_thr:.6f} (recall={val_recall:.3f}).")

# 3) Metrics
def summarize(y_true, p, thr):
    y_hat = (p >= thr).astype(int)
    return {
        "AUC": float(roc_auc_score(y_true, p)),
        "AP": float(average_precision_score(y_true, p)),
        "precision": float(precision_score(y_true, y_hat, zero_division=0)),
        "recall": float(recall_score(y_true, y_hat, zero_division=0)),
        "f1": float(f1_score(y_true, y_hat, zero_division=0)),
        "threshold": float(thr),
        "confusion_matrix": confusion_matrix(y_true, y_hat).tolist()
    }

val_metrics = summarize(y_val, p_val, best_thr)
test_metrics = summarize(y_te,  p_te,  best_thr)

# 4) Per-complaint breakdown (by Leitsymptom code)
inv_map = {v:k for k,v in coding_maps.get("complaint_to_code", {}).items()}
df_eval = df_test.copy()
df_eval["p_cal"] = p_te
df_eval["y_hat"] = (p_te >= best_thr).astype(int)
df_eval["complaint"] = df_eval["Leitsymptom"].map(inv_map).fillna("UNK")

per_c = {}
for c, g in df_eval.groupby("complaint"):
    if len(g) < 20: continue
    y_true = g[TARGET].to_numpy(int)
    y_hat  = g["y_hat"].to_numpy(int)
    p      = g["p_cal"].to_numpy(float)
    try:
        auc = roc_auc_score(y_true, p)
        ap  = average_precision_score(y_true, p)
    except Exception:
        auc, ap = float("nan"), float("nan")
    far = float((((y_hat==1)&(y_true==0)).sum())/max(1,(y_true==0).sum()))
    per_c[c] = {
        "n": int(len(g)),
        "AUC": float(auc) if auc==auc else None,
        "AP": float(ap)   if ap==ap   else None,
        "precision": float(precision_score(y_true, y_hat, zero_division=0)),
        "recall": float(recall_score(y_true, y_hat, zero_division=0)),
        "f1": float(f1_score(y_true, y_hat, zero_division=0)),
        "false_alarm_rate": far
    }

# 5) Persist artifacts
import json, joblib, os
joblib.dump(pipe, f"{OUT_DIR}/ed_phase2_model_synth100.joblib")
joblib.dump(iso,  f"{OUT_DIR}/ed_phase2_isotonic_synth100.joblib")
Path(f"{OUT_DIR}/ed_phase2_threshold_synth100.txt").write_text(str(best_thr))
Path(f"{OUT_DIR}/ed_phase2_report_synth100.json").write_text(json.dumps({"val":val_metrics,"test":test_metrics,"per_complaint":per_c}, indent=2))
df_eval[["p_cal","y_hat","Gate_Pos","Leitsymptom"]].to_csv(f"{OUT_DIR}/ed_phase2_test_predictions_synth100.csv", index=False)

print("== Validation (real) =="); print(val_metrics)
print("== Test (real) =="); print(test_metrics)
print("Per-complaint (n>=20):", per_c)
print("Artifacts written to:", OUT_DIR)


OK: isotonic fitted & monotone.
OK: recall floor met on validation @thr=0.000000 (recall=1.000).
== Validation (real) ==
{'AUC': 0.539642229800048, 'AP': 0.13867638290159945, 'precision': 0.12378472222222223, 'recall': 1.0, 'f1': 0.2202997064730419, 'threshold': 0.0, 'confusion_matrix': [[0, 5047], [0, 713]]}
== Test (real) ==
{'AUC': 0.5130320039791777, 'AP': 0.1709393310540161, 'precision': 0.14288194444444444, 'recall': 1.0, 'f1': 0.2500379766064104, 'threshold': 0.0, 'confusion_matrix': [[0, 4937], [0, 823]]}
Per-complaint (n>=20): {'abd_pain': {'n': 1548, 'AUC': 0.49588477366255146, 'AP': 0.05996002664890073, 'precision': 0.05813953488372093, 'recall': 1.0, 'f1': 0.10989010989010987, 'false_alarm_rate': 1.0}, 'chest_pain': {'n': 792, 'AUC': 0.5068225930730441, 'AP': 0.22228059380799048, 'precision': 0.2058080808080808, 'recall': 1.0, 'f1': 0.3413612565445026, 'false_alarm_rate': 1.0}, 'dyspnea': {'n': 828, 'AUC': 0.7399367419846634, 'AP': 0.3881666910652418, 'precision': 0.0688405